In [0]:
# [MVR-REF: 2.3] - SILVER LAYER REMEDIATION (DATA INTEGRITY CHECK)
# LAYER: Silver (refined data)
# PURPOSE: Verification of transformation logic for volatility clusters
# REQUIREMENT: SR 11-7, MRM Practice Note (May 2019, American Academy of Actuaries)
# LOGIC: Validation of log-return transformation accuracy.

# Silver layer return calculation fix

%pip install arch yfinance

import pandas as pd
import numpy as np
import yfinance as yf
from arch import arch_model
from statsmodels.stats.diagnostic import acorr_ljungbox 
from pyspark.sql import functions as F
from pyspark.sql.window import Window

bronze_df = spark.table("bronze_market_data")

# 1. Strict deduplication
window_spec = Window.partitionBy("Date").orderBy("Date")
deduped_df = bronze_df.withColumn("row_num", F.row_number().over(window_spec)) \
    .filter(F.col("row_num") == 1).drop("row_num")

# 2. Filter for validity.
# Ensure Close is a positive number to avoid log(-inf) or log(0)
valid_df = deduped_df.filter(F.col("Close") > 0.01)

# 3. Calculate returns with a null check.
window_lag = Window.orderBy("Date")
silver_df = valid_df.withColumn("prev_close", F.lag("Close").over(window_lag))

# A conditional is used to ensure no division by zero or null.
silver_df = silver_df.withColumn("return", F.when(F.col("prev_close")
                                                  .isNotNull() & (F.col("prev_close") > 0),
                                                  F.log(F.col("Close") / F.col("prev_close")))
                                 .otherwise(F.lit(None)))


# 4. Final cleanup: remove the extreme artifacts
# (Real S&P 500 returns never exceed 20% in a single day.)
# Anything beyond 0.5 is a data error.
silver_df = silver_df.filter(F.abs(F.col("return")) < 0.5)

# 5. Save
silver_df.select("Date", "Close", "return") \
    .write.mode("overwrite") \
        .format("delta") \
            .option("overwriteSchema", "true") \
                .saveAsTable("silver_market_data")

print("Silver Layer: Massive artifacts removed. Returns now within realistic bounds.")